<a href="https://colab.research.google.com/github/reshmiee/Disha/blob/main/person3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q statsmodels

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED, NBOOT = 0, 10000
PILOT_THRESHOLDS = {'10%': 0.59491, '20%': 0.50944, '30%': 0.46514, '50%': 0.4066, '70%': 0.33721}
FEATURE_DIFFS = ['tokens_diff_en_minus_hi', 'words_diff_en_minus_hi', 'fert_diff_en_minus_hi']

# ---------- small utilities ----------
def save_csv(df, out_dir, name):
    """Save as name_v1.csv, name_v2.csv ... never overwriting an earlier file."""
    os.makedirs(out_dir, exist_ok=True)
    v = 1
    while os.path.exists(f'{out_dir}/{name}_v{v}.csv'):
        v += 1
    path = f'{out_dir}/{name}_v{v}.csv'
    df.to_csv(path, index=False)
    print('  saved', path)
    return path

def save_fig(fig, out_dir, name):
    os.makedirs(out_dir, exist_ok=True)
    v = 1
    while os.path.exists(f'{out_dir}/{name}_v{v}.png'):
        v += 1
    path = f'{out_dir}/{name}_v{v}.png'
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print('  saved', path)

def banner(title, confirmatory):
    tag = 'CONFIRMATORY (in the frozen plan)' if confirmatory else 'EXPLORATORY (not in the frozen plan)'
    print(f'\n=== {title}: {tag} ===')

# ---------- loading and checks ----------
def load_inputs(scores_path, feats_path=None):
    s = pd.read_csv(scores_path)
    need = ['pair_id', 'english_score', 'hinglish_score']
    miss = [c for c in need if c not in s.columns]
    assert not miss, f'scores file is missing columns: {miss}'
    assert s['pair_id'].is_unique, 'duplicate pair_id in scores file'
    sc = ['english_score', 'hinglish_score']
    assert s[sc].notna().all().all(), 'missing scores'
    assert s[sc].apply(lambda c: c.between(0, 1)).all().all(), 'a score is outside 0-1'
    s['diff_en_minus_hi'] = s['english_score'] - s['hinglish_score']   # always recomputed: EN minus HI
    if feats_path:
        f = pd.read_csv(feats_path)
        assert f['pair_id'].is_unique, 'duplicate pair_id in features file'
        extra = [c for c in f.columns if c not in s.columns or c == 'pair_id']
        s = s.merge(f[extra], on='pair_id', how='left', validate='one_to_one')
        assert len(s) == len(f), 'scores and features do not have the same pairs'
    print(f'loaded {len(s)} pairs; columns: {list(s.columns)}')
    return s

def load_thresholds(config_path):
    if config_path and os.path.exists(config_path):
        c = json.load(open(config_path))
        try:
            return {'10%': c['threshold_10pct'], '20%': c['threshold_20pct'], '30%': c['threshold_30pct'],
                    '50%': c['threshold'], '70%': c['threshold_70pct']}
        except KeyError:
            print('config is missing some thresholds; using the pilot thresholds')
    else:
        print('no config file; using the pilot thresholds')
    return dict(PILOT_THRESHOLDS)

# ---------- core statistics ----------
def orth_slope(x, y):
    xc, yc = x - x.mean(), y - y.mean()
    sxx, syy, sxy = (xc**2).sum(), (yc**2).sum(), (xc * yc).sum()
    if sxy == 0:
        return np.nan
    return (syy - sxx + np.sqrt((syy - sxx)**2 + 4 * sxy**2)) / (2 * sxy)

def crossover(x, y):
    s, i = np.polyfit(x, y, 1)
    return i / (1 - s) if abs(1 - s) > 1e-6 else np.nan     # score of x where y equals x

def pair_stats(x, y, label):
    """Compare two paired score vectors (x = English, y = the other version)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x)
    idx = np.random.default_rng(SEED).integers(0, n, size=(NBOOT, n))   # resample PAIRS
    ratio = x[idx].std(1, ddof=1) / y[idx].std(1, ddof=1)
    osl = np.array([orth_slope(x[b], y[b]) for b in idx])
    cr = np.array([crossover(x[b], y[b]) for b in idx])
    r, _ = stats.pearsonr(x + y, x - y)                  # Pitman-Morgan
    t = r * np.sqrt((n - 2) / (1 - r**2))
    pm_p = 2 * stats.t.sf(abs(t), n - 2)
    q = lambda a: np.nanpercentile(a, [2.5, 97.5])
    return {'comparison': label, 'n': n, 'mean_en': x.mean(), 'mean_other': y.mean(),
            'mean_diff_en_minus_other': (x - y).mean(), 'sd_en': x.std(ddof=1), 'sd_other': y.std(ddof=1),
            'sd_ratio': x.std(ddof=1) / y.std(ddof=1), 'sd_ratio_lo': q(ratio)[0], 'sd_ratio_hi': q(ratio)[1],
            'pitman_morgan_r': r, 'pitman_morgan_p': pm_p,
            'ols_slope': np.polyfit(x, y, 1)[0], 'orth_slope': orth_slope(x, y),
            'orth_lo': q(osl)[0], 'orth_hi': q(osl)[1],
            'crossover': crossover(x, y), 'cross_lo': q(cr)[0], 'cross_hi': q(cr)[1]}

# ---------- analyses ----------
def main_result(df):
    d = df['diff_en_minus_hi'].to_numpy()
    n = len(d)
    boots = d[np.random.default_rng(SEED).integers(0, n, (NBOOT, n))].mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    t = stats.ttest_rel(df['english_score'], df['hinglish_score'])      # two-sided
    w = stats.wilcoxon(d)                                               # two-sided
    return pd.DataFrame([{'n': n, 'mean_en': df.english_score.mean(), 'mean_hi': df.hinglish_score.mean(),
        'mean_diff_en_minus_hi': d.mean(), 'ci_lo': lo, 'ci_hi': hi, 'dz': d.mean() / d.std(ddof=1),
        'en_higher_in': int((d > 0).sum()), 'hi_higher_in': int((d < 0).sum()),
        't_p': t.pvalue, 'wilcoxon_p': w.pvalue}])

def routing_table(df, thr, primary='50%'):
    rows = []
    for k, t in sorted(thr.items(), key=lambda kv: float(kv[0].rstrip('%'))):
        en, hi = df['english_score'] >= t, df['hinglish_score'] >= t
        both, neither = int((en & hi).sum()), int((~en & ~hi).sum())
        eo, ho = int((en & ~hi).sum()), int((~en & hi).sum())
        p = mcnemar([[both, eo], [ho, neither]], exact=True).pvalue
        rows.append({'nominal_strong': k, 'threshold': round(t, 5), 'EN_strong_pct': round(100 * en.mean(), 1),
                     'HI_strong_pct': round(100 * hi.mean(), 1), 'EN_only': eo, 'HI_only': ho, 'p_raw': p})
    tab = pd.DataFrame(rows)
    tab['p_holm'] = multipletests(tab['p_raw'], method='holm')[1]
    tab['primary'] = tab['nominal_strong'].eq(primary)
    return tab

def bland_altman(df, out_dir):
    x, y = df['english_score'].to_numpy(), df['hinglish_score'].to_numpy()
    avg, diff = (x + y) / 2, x - y
    sl, ic = np.polyfit(avg, diff, 1)
    r, p = stats.pearsonr(avg, diff)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(avg, diff, s=18, alpha=0.7)
    m, sd = diff.mean(), diff.std(ddof=1)
    ax.axhline(m, color='k', lw=1, label=f'mean diff {m:.3f}')
    ax.axhline(m + 1.96 * sd, color='gray', ls='--', lw=1, label='limits of agreement')
    ax.axhline(m - 1.96 * sd, color='gray', ls='--', lw=1)
    xs = np.linspace(avg.min(), avg.max(), 50)
    ax.plot(xs, sl * xs + ic, color='C3', lw=1.5, label=f'trend (slope {sl:.2f})')
    ax.axhline(0, color='lightgray', lw=0.8)
    ax.set_xlabel('Average of English and Hinglish score'); ax.set_ylabel('English minus Hinglish')
    ax.set_title('Bland-Altman: score difference vs average score'); ax.legend(fontsize=8)
    save_fig(fig, out_dir, 'fig_bland_altman')
    return {'slope': sl, 'r': r, 'p': p}

def group_breakdown(df, col):
    rows = []
    for g, sub in df.groupby(col):
        d = sub['diff_en_minus_hi']
        n = len(d)
        se = d.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
        tcrit = stats.t.ppf(0.975, n - 1) if n > 1 else np.nan
        wp = stats.wilcoxon(d).pvalue if n >= 6 and (d != 0).any() else np.nan
        rows.append({col: g, 'n': n, 'mean_en': sub.english_score.mean(), 'mean_hi': sub.hinglish_score.mean(),
                     'mean_diff': d.mean(), 'ci_lo': d.mean() - tcrit * se, 'ci_hi': d.mean() + tcrit * se,
                     'sd_en': sub.english_score.std(ddof=1), 'sd_hi': sub.hinglish_score.std(ddof=1),
                     'wilcoxon_p': wp})
    return pd.DataFrame(rows)

def two_way_model(df, cols):
    """Type II ANOVA of the score difference on task type, difficulty, writer.
    Refuses to print F-values when the factors overlap (rank-deficient design)."""
    cols = [c for c in cols if c in df.columns and df[c].nunique() > 1]
    if not cols:
        return None
    f = 'diff_en_minus_hi ~ ' + ' + '.join(f'C({c})' for c in cols)
    model = smf.ols(f, data=df)
    X = model.exog
    rank = np.linalg.matrix_rank(X)
    if rank < X.shape[1]:
        print(f'  ANOVA NOT SHOWN: the design is rank-deficient (rank {rank} of {X.shape[1]} columns).')
        print(f'  The factors {cols} overlap, so their effects cannot be separated.')
        if len(cols) >= 2:
            print(f'  Pairs per cell, {cols[0]} by {cols[1]} (zeros = combinations that never occur):')
            print(pd.crosstab(df[cols[0]], df[cols[1]]).to_string())
        print('  Fix the design (cross the factors) before interpreting any factor effect.')
        return None
    a = sm.stats.anova_lm(model.fit(), typ=2).reset_index().rename(columns={'index': 'term'})
    return a

def ridge_features(df):
    cols = [c for c in FEATURE_DIFFS if c in df.columns]
    if len(cols) < 2 or df[cols].isna().any().any():
        print('  feature columns missing or incomplete; skipping ridge')
        return None, None
    X, y = df[cols].astype(float), df['diff_en_minus_hi'].to_numpy()
    Z = pd.DataFrame(StandardScaler().fit_transform(X), columns=cols)
    Zc = sm.add_constant(Z)
    vif = pd.DataFrame({'feature': cols,
                        'VIF': [variance_inflation_factor(Zc.values, i + 1) for i in range(len(cols))]})
    ridge = RidgeCV(alphas=np.logspace(-2, 3, 30)).fit(Z, y)
    cv = cross_val_score(make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))), X, y,
                         cv=KFold(5, shuffle=True, random_state=SEED), scoring='r2')
    res = pd.DataFrame({'feature': cols, 'standardized_coef': ridge.coef_,
                        'spearman_with_diff': [stats.spearmanr(X[c], y)[0] for c in cols],
                        'spearman_p': [stats.spearmanr(X[c], y)[1] for c in cols]})
    res['ridge_alpha'] = ridge.alpha_
    res['cv_r2_mean'] = cv.mean()
    res['cv_r2_sd'] = cv.std()
    return res.merge(vif, on='feature'), vif

def charts(df, out_dir):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    bins = np.linspace(0, 1, 26)
    ax.hist(df.english_score, bins=bins, alpha=0.6, label='English')
    ax.hist(df.hinglish_score, bins=bins, alpha=0.6, label='Hinglish')
    ax.set_xlabel('Router score'); ax.set_ylabel('Pairs'); ax.set_title('Score distributions'); ax.legend()
    save_fig(fig, out_dir, 'fig_score_distributions')
    for col in ['task_type', 'difficulty', 'written_by']:
        if col in df.columns and df[col].nunique() > 1:
            groups = [(str(g), s['diff_en_minus_hi'].to_numpy()) for g, s in df.groupby(col)]
            fig, ax = plt.subplots(figsize=(max(5, 0.9 * len(groups) + 2), 4))
            ax.boxplot([g[1] for g in groups], tick_labels=[g[0] for g in groups])
            ax.axhline(0, color='lightgray', lw=0.8)
            ax.set_ylabel('English minus Hinglish'); ax.set_title(f'Score difference by {col}')
            plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
            save_fig(fig, out_dir, f'fig_diff_by_{col}')
    fcols = [c for c in FEATURE_DIFFS if c in df.columns]
    if fcols:
        fig, axes = plt.subplots(1, len(fcols), figsize=(4.2 * len(fcols), 3.8))
        for ax, c in zip(np.atleast_1d(axes), fcols):
            ax.scatter(df[c], df['diff_en_minus_hi'], s=14, alpha=0.7)
            ax.set_xlabel(c); ax.set_ylabel('score diff (EN - HI)')
        save_fig(fig, out_dir, 'fig_feature_scatter')

def control_table(path, pairs):
    """pairs = [(other_column, label), ...] compared against english_score in that file."""
    c = pd.read_csv(path)
    rows = []
    for col, label in pairs:
        if col in c.columns and c[col].notna().all():
            rows.append(pair_stats(c['english_score'] if 'english_score' in c.columns else c[pairs[0][0]], c[col], label))
    return pd.DataFrame(rows)

# ---------- the one call ----------
def run_analysis(scores_path, out_dir, feats_path=None, config_path=None, triple_path=None,
                 scrambled_path=None, langpair_path=None, primary='50%',
                 confirmatory=('main', 'routing', 'compression')):
    """confirmatory = names of the analyses listed in the frozen plan; others print as EXPLORATORY."""
    C = lambda name: name in confirmatory
    df = load_inputs(scores_path, feats_path)
    thr = load_thresholds(config_path)
    out = {}

    banner('1. Main result (EN minus HI)', C('main'))
    out['main'] = main_result(df); print(out['main'].round(4).to_string(index=False))
    save_csv(out['main'], out_dir, 'p3_main_result')

    banner('2. Routing at five thresholds, Holm-corrected', C('routing'))
    out['routing'] = routing_table(df, thr, primary); print(out['routing'].round(4).to_string(index=False))
    save_csv(out['routing'], out_dir, 'p3_routing_holm')

    banner('3. Compression: spread, slope, crossover', C('compression'))
    out['compression'] = pd.DataFrame([pair_stats(df.english_score, df.hinglish_score, 'English vs Hinglish')])
    print(out['compression'].round(3).T.to_string(header=False))
    save_csv(out['compression'], out_dir, 'p3_compression')
    out['bland_altman'] = bland_altman(df, out_dir)
    print('  Bland-Altman diff vs average:', {k: round(v, 4) for k, v in out['bland_altman'].items()},
          '(same information as Pitman-Morgan; report as ONE finding)')
    print('  pilot to compare: SD ratio 1.81, orthogonal slope 0.51, crossover 0.477 (CI 0.461-0.494)')

    banner('4. Breakdowns by task type, difficulty, writer', C('breakdowns'))
    for col in ['task_type', 'difficulty', 'written_by']:
        if col in df.columns:
            t = group_breakdown(df, col); out[f'by_{col}'] = t
            print(f'\n  by {col}:'); print(t.round(3).to_string(index=False))
            save_csv(t, out_dir, f'p3_by_{col}')
    tw = two_way_model(df, ['task_type', 'difficulty', 'written_by'])
    if tw is not None:
        out['two_way'] = tw; print('\n  two-way model (type II ANOVA on the score difference):')
        print(tw.round(4).to_string(index=False)); save_csv(tw, out_dir, 'p3_two_way_anova')

    banner('5. Controls', C('controls'))
    if triple_path and os.path.exists(triple_path):
        t = control_table(triple_path, [('hinglish_score', 'English vs current Hinglish'),
                                        ('hinglish_translated_score', 'English vs translated Romanized'),
                                        ('hinglish_devanagari_score', 'English vs Devanagari')])
        out['triple'] = t; print(t[['comparison', 'n', 'sd_en', 'sd_other', 'sd_ratio', 'sd_ratio_lo', 'sd_ratio_hi',
                                    'orth_slope', 'orth_lo', 'orth_hi']].round(3).to_string(index=False))
        save_csv(t, out_dir, 'p3_control_triple')
    if scrambled_path and os.path.exists(scrambled_path):
        s = pd.read_csv(scrambled_path)
        t = pd.DataFrame([pair_stats(s.english_scrambled_score, s.hinglish_scrambled_score,
                                     'scrambled: English vs Hinglish')])
        out['scrambled'] = t; print(t[['comparison', 'n', 'sd_en', 'sd_other', 'sd_ratio', 'sd_ratio_lo',
                                       'sd_ratio_hi', 'orth_slope']].round(3).to_string(index=False))
        save_csv(t, out_dir, 'p3_control_scrambled')
    if langpair_path and os.path.exists(langpair_path):
        t = control_table(langpair_path, [('other_score', 'English vs other language pair')])
        out['langpair'] = t; print(t[['comparison', 'n', 'sd_en', 'sd_other', 'sd_ratio', 'sd_ratio_lo',
                                      'sd_ratio_hi', 'orth_slope']].round(3).to_string(index=False))
        save_csv(t, out_dir, 'p3_control_langpair')
    if not any([triple_path, scrambled_path, langpair_path]):
        print('  no control files given; skipped')

    banner('6. Ridge regression on feature differences (+ VIF)', C('ridge'))
    res, vif = ridge_features(df)
    if res is not None:
        out['ridge'] = res; print(res.round(4).to_string(index=False)); save_csv(res, out_dir, 'p3_ridge_vif')
        print('  VIF above 5-10 means the features overlap; read coefficients with caution.')

    print('\n=== 7. Charts ===')
    charts(df, out_dir)
    print('\nDone. Compare section 3 with the pilot values and fill the report tables from the saved CSVs.')
    return out

In [ ]:
D = '/content/drive/MyDrive/ML_Research'
results = run_analysis(
    scores_path = f'{D}/_dryrun/scores/scores_confirmatory_v1.csv',
    feats_path  = f'{D}/_dryrun/features/features_confirmatory_v1.csv',
    config_path = f'{D}/_dryrun/results/config_conf_v1.json',
    out_dir     = f'{D}/_dryrun/results/person3',
)

loaded 80 pairs; columns: ['pair_id', 'english_score', 'hinglish_score', 'diff_en_minus_hi', 'task_type', 'difficulty', 'written_by', 'english_words', 'english_tokens', 'english_fertility', 'hinglish_words', 'hinglish_tokens', 'hinglish_fertility', 'tokens_diff_en_minus_hi', 'words_diff_en_minus_hi', 'fert_diff_en_minus_hi']

=== 1. Main result (EN minus HI): CONFIRMATORY (in the frozen plan) ===
 n  mean_en  mean_hi  mean_diff_en_minus_hi   ci_lo   ci_hi      dz  en_higher_in  hi_higher_in    t_p  wilcoxon_p
80   0.4104   0.4451                -0.0347 -0.0543 -0.0161 -0.3928            30            50 0.0007       0.001
  saved /content/drive/MyDrive/ML_Research/_dryrun/results/person3/p3_main_result_v1.csv

=== 2. Routing at five thresholds, Holm-corrected: CONFIRMATORY (in the frozen plan) ===
nominal_strong  threshold  EN_strong_pct  HI_strong_pct  EN_only  HI_only  p_raw  p_holm  primary
           10%     0.5949            8.8            1.2        6        0 0.0312  0.0938    F

/tmp/ipykernel_4382/646545973.py:175: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  m = smf.ols(f, data=df).fit()


                feature  standardized_coef  spearman_with_diff  spearman_p  ridge_alpha  cv_r2_mean  cv_r2_sd     VIF
tokens_diff_en_minus_hi            -0.0521             -0.1533      0.1747       1.1721     -0.1932    0.1519  7.7736
 words_diff_en_minus_hi             0.0552             -0.0590      0.6034       1.1721     -0.1932    0.1519 10.5564
  fert_diff_en_minus_hi             0.0405              0.0324      0.7753       1.1721     -0.1932    0.1519  3.3044
  saved /content/drive/MyDrive/ML_Research/_dryrun/results/person3/p3_ridge_vif_v1.csv
  VIF above 5-10 means the features overlap; read coefficients with caution.

=== 7. Charts ===
  saved /content/drive/MyDrive/ML_Research/_dryrun/results/person3/fig_score_distributions_v1.png
  saved /content/drive/MyDrive/ML_Research/_dryrun/results/person3/fig_diff_by_task_type_v1.png
  saved /content/drive/MyDrive/ML_Research/_dryrun/results/person3/fig_diff_by_difficulty_v1.png
  saved /content/drive/MyDrive/ML_Research/_dryrun/res